In [33]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import xml.etree.ElementTree as ET

### Import Drugbank

In [34]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [35]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [36]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Import genetic results

In [37]:
hpv_pos_direct_genes = pd.read_csv('Results/HPV Positive validated genes.csv')
hpv_neg_direct_genes = pd.read_csv('Results/HPV Negative validated genes.csv')

In [38]:
hpv_pos_direct_genes

,Unnamed: 0,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
0,0,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
1,1,CLDN1,AMPLIFICATION,6.591105366473553e-53,0.0115776022868357,CLDN1,"15170668, 17091452",2.0
2,2,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0
3,3,FANCB,DELETION,8.234936784924659e-31,0.0203434456037747,FANCB,17409780,1.0
4,4,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
5,5,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,RFC4,16467079,1.0
6,6,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0
7,7,STS,DELETION,8.234936784924659e-31,0.0203434456037747,STS,11746990,1.0
8,8,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,TLR7,17201162,1.0


In [39]:
### connect the direct genes to drug bank
def connect_genes_to_drugbank(gene_df, drugbank_df = Drug_bank):
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()  # Normalize gene names to uppercase
    drugbank_df['gene'] = drugbank_df['gene'].str.upper()  # Normalize gene names in drug bank to uppercase
    connected_drugs = []
    for i, gene in tqdm(enumerate(gene_df['gene_name'])):
        drugs_for_gene = drugbank_df[drugbank_df['gene'] == gene]['drug'].unique().tolist()
        connected_drugs.append(drugs_for_gene)
    gene_df['connected_drugs'] = connected_drugs
    return gene_df

In [40]:
hpv_pos_direct_genes_with_drugbank = connect_genes_to_drugbank(hpv_pos_direct_genes, Drug_bank)
hpv_neg_direct_genes_with_drugbank = connect_genes_to_drugbank(hpv_neg_direct_genes, Drug_bank)

0it [00:00, ?it/s]

9it [00:00, 1297.43it/s]
31it [00:00, 1396.00it/s]


In [41]:
hpv_pos_direct_genes_with_drugbank

,Unnamed: 0,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES,connected_drugs
0,0,BCL6,AMPLIFICATION,1.1060251600493613e-54,0.0115776022868357,BCL6,"11224600, 11420458, 14685876, 17429099",4.0,[]
1,1,CLDN1,AMPLIFICATION,6.591105366473553e-53,0.0115776022868357,CLDN1,"15170668, 17091452",2.0,[]
2,2,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0,[]
3,3,FANCB,DELETION,8.234936784924659e-31,0.0203434456037747,FANCB,17409780,1.0,[]
4,4,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0,"[ATP, Caffeine, XL765, Wortmannin, Buparlisib,..."
5,5,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,RFC4,16467079,1.0,[]
6,6,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,SOX2,15942670,1.0,[]
7,7,STS,DELETION,8.234936784924659e-31,0.0203434456037747,STS,11746990,1.0,"[Irosustat, 2-Amino-3-Oxo-4-Sulfo-Butyric Acid..."
8,8,TLR7,DELETION,8.234936784924659e-31,0.0203434456037747,TLR7,17201162,1.0,"[Imiquimod, Hydroxychloroquine, Isatoribine, A..."


In [42]:
hpv_neg_direct_genes_with_drugbank

,Unnamed: 0,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES,connected_drugs
0,0,ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343,ABCC5,18359711,1.0,"[Curcumin, Curcumin sulfate]"
1,1,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,BCL6,"11224600, 11420458, 14685876, 17429099",4.0,[]
2,2,CASP8,SOMATIC,1.4328145167446222e-45,0.005960073448722,CASP8,16857411,1.0,"[Trichostatin A, AN-9, Bryostatin 1, Bardoxolo..."
3,3,CDH7,SOMATIC,0.0040235357729719,0.005960073448722,CDH7,12118382,1.0,[]
4,4,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722",CDKN2A,"11309301, 11720478, 12459644, 12632081, 126844...",16.0,[]
5,5,CDKN2B,DELETION,0.0,0.0191774659792392,CDKN2B,"12632081, 17673925",2.0,[]
6,6,CLDN1,AMPLIFICATION,4.2419726156021995e-197,0.0043623451388343,CLDN1,"15170668, 17091452",2.0,[]
7,7,COL11A1,SOMATIC,1.5777323720244649e-15,0.005960073448722,COL11A1,15023835,1.0,[]
8,8,CSMD3,SOMATIC,5.936981595700065e-36,0.005960073448722,CSMD3,12906867,1.0,[]
9,9,DCC,SOMATIC,0.0165073700948121,0.005960073448722,DCC,"11603424, 12374684, 16400327",3.0,[]


In [43]:
for drugs in hpv_pos_direct_genes_with_drugbank['connected_drugs']:
    for drug in drugs:
        if 'HEPARIN' in drug.upper():
            print(drug)

### Import EHR drug results

In [44]:
ehr_hpv_pos_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_positive_ml_drug_xgb_results.csv')
ehr_hpv_neg_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_negative_ml_drug_xgb_results.csv')

### replace "_aspirin" in ehr_hpv_neg_drugs with "_acetylsalicylic acid"
ehr_hpv_neg_drugs['feature'] = ehr_hpv_neg_drugs['feature'].str.replace('_aspirin', '_acetylsalicylic acid')

### Combine EHR and genetic results

In [45]:
def connect_ehr_and_genetic(ehr_drug_df, genetic_gene_df):
    ehr_drug_df['feature'] = ehr_drug_df['feature'].str.upper()  # Normalize drug names to uppercase
    combined_results = []
    for i, row in ehr_drug_df.iterrows():
        drug_name = row['feature'][1:] # remove the _ prefix
        drug_name = drug_name.upper()  # Normalize to uppercase for matching
        # Check if this drug is connected to any genes in the genetic results
        for j, gene_row in genetic_gene_df.iterrows():
            if drug_name in [d.upper() for d in gene_row['connected_drugs']]:
                combined_results.append({
                    'drug': drug_name,
                    'ehr_xgboost_importance': row['xgb_importance'],
                    'ehr_log_odds_ratio': row['log_odds_ratio'],
                    'genetic_gene': gene_row['gene_name'],
                    'genetic_empirical_q_value': gene_row['empirical_q_value'],
                    'genetic_q_value': gene_row['q_value']
                })
    return pd.DataFrame(combined_results)

In [46]:
connect_ehr_and_genetic(ehr_hpv_pos_drugs, hpv_pos_direct_genes_with_drugbank)

""


In [47]:
connect_ehr_and_genetic(ehr_hpv_neg_drugs, hpv_neg_direct_genes_with_drugbank)

,drug,ehr_xgboost_importance,ehr_log_odds_ratio,genetic_gene,genetic_empirical_q_value,genetic_q_value
0,ACETYLSALICYLIC ACID,0.003261,0.616444,TP53,0.005960073448722,0.0


In [48]:
'LEVOTHYROXINE' in [d.upper() for d in hpv_pos_direct_genes_with_drugbank['connected_drugs'].explode().dropna().unique()]

False